# RISKTWIN · Pipeline Supabase: Dataset_Limpio y dimensiones H, E, V — Vía Perimetral

Ejecuta **Entorno de ejecución → Ejecutar todas**. El notebook trabaja solo, de principio a fin:

| Fase | Qué hace | Tabla de Supabase |
|---|---|---|
| 1 | Sube el Excel `DatasetPerimetral_24_26.xlsx` a Supabase (solo la primera vez) | `dataset_perimetral_raw` |
| 2 | Trae los datos crudos desde Supabase al notebook | — |
| 3 | Limpia: corrige columnas desplazadas, normaliza faltantes, elimina duplicados | — |
| 4 | Sube el resultado | `Dataset_Limpio` |
| 5 | Vuelve a traer `Dataset_Limpio` desde Supabase | — |
| 6 | Arma la red vial: tramos de 500 m (aristas) y sus nodos | — |
| 7 | Dimensión **H** (Peligrosidad) + mapa | `datasetDim_H` |
| 8 | Dimensión **E** (Exposición) + mapa | `datasetDim_E` |
| 9 | Dimensión **V** (Vulnerabilidad) + mapa | `datasetDim_V` |
| 10 | Lee las tres dimensiones desde Supabase y las integra en un mapa interactivo | — |

### APIs que usa
- **Supabase REST (PostgREST)**, mediante la librería `supabase` de Python:
  - URL: `https://myvmerqmopraucubddrx.supabase.co`.
  - **Lectura:** con la clave pública (*publishable*), que ya viene en el código.
  - **Escritura:** con la **secret key**. Guárdala en 🔑 *Secrets* de Colab con el nombre `SUPABASE_SECRET_KEY`. La encuentras en Supabase → *Project Settings → API Keys* (`sb_secret_…` o la *service_role*). Nunca la pegues en el código.
- **Overpass API (OpenStreetMap)**: descarga el trazado de la Vía Perimetral. El resultado se guarda en caché local para no descargarlo de nuevo.

## 0. Instalación y configuración

In [ ]:
import json, re, time, os
import numpy as np, pandas as pd, geopandas as gpd
from dotenv import load_dotenv, find_dotenv
from supabase import create_client

# --- conexión a Supabase (.env) ---
load_dotenv(find_dotenv(), override=True)

SUPABASE_URL         = os.getenv('SUPABASE_URL')
SUPABASE_PUBLISHABLE = os.getenv('SUPABASE_PUBLISHABLE')
_SK                  = os.getenv('SUPABASE_KEY')

if not SUPABASE_URL or not SUPABASE_PUBLISHABLE:
    raise RuntimeError('Faltan SUPABASE_URL o SUPABASE_PUBLISHABLE en el archivo .env')

sb_lectura   = create_client(SUPABASE_URL, SUPABASE_PUBLISHABLE)
sb_escritura = create_client(SUPABASE_URL, _SK) if _SK else None
print('Supabase → Lectura: OK · Escritura:', 'OK' if sb_escritura else 'SOLO LECTURA (falta SUPABASE_KEY)')

# --- mapa base ---
CARTO_KEY   = 'cb1_5do9_1_90e38ab2e2043b7ab55c5b13'
TILES_CARTO = f'https://basemaps.cartocdn.com/rastertiles/light_all/{{z}}/{{x}}/{{y}}.png?key={CARTO_KEY}'
ATTR_CARTO  = '© OpenStreetMap contributors © CARTO'

# --- parámetros del pipeline ---
RUTA_EXCEL        = 'DatasetPerimetral_24_26.xlsx'
FORZAR_RECARGA    = False        # True = vuelve a subir el Excel crudo aunque ya esté en Supabase
LONGITUD_TRAMO_M  = 500
DISTANCIA_MAX_M   = 250          # siniestros más lejos del eje no se asignan a un tramo
TOLERANCIA_NODO_M = 30           # extremos de tramo a menos de esto son el mismo nodo
SUAVIZADO         = 5
UMBRALES          = (0.33, 0.66) # Baja < 0.33 ≤ Media < 0.66 ≤ Alta
PESOS             = {'H': 1/3, 'E': 1/3, 'V': 1/3}
HORAS_PICO        = {6, 7, 8, 17, 18, 19}
CRS_METROS        = 32717        # UTM 17S
BBOX_GYE          = (-2.40, -80.10, -1.95, -79.75)   # lat_min, lon_min, lat_max, lon_max
LOTE              = 500          # filas por envío a Supabase

VEH = ['automovil', 'bicicleta', 'bus', 'camion', 'camioneta', 'emergencias', 'especial', 'furgoneta',
       'motocicleta', 'no_identificado', 'scooter_electrico', 'tricimoto', 'vehiculo_deportivo_utilitario']

Lectura: OK · Escritura: OK (secret key encontrada)


In [12]:
import subprocess, sys, os, ssl
from urllib.parse import urlsplit, urlunsplit

for pkg in ("sqlalchemy", "pg8000"):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine, text

load_dotenv(find_dotenv(), override=True)
raw_url = os.getenv("DATABASE_URL")

parts = urlsplit(raw_url)
db_url = urlunsplit(("postgresql+pg8000", parts.netloc, parts.path, "", ""))

ssl_ctx = ssl.create_default_context()
ssl_ctx.check_hostname = False
ssl_ctx.verify_mode = ssl.CERT_NONE

engine = create_engine(db_url, connect_args={"ssl_context": ssl_ctx})

SQL_TABLAS = """
CREATE TABLE IF NOT EXISTS public.dataset_perimetral_raw (
    id_fila BIGSERIAL PRIMARY KEY, hoja_origen TEXT, id_excel BIGINT, anio INT,
    siniestros TEXT, lesionados INT, fallecidos INT, latitud_y DOUBLE PRECISION,
    longitud_x DOUBLE PRECISION, direccion TEXT, nombre_de_la_via TEXT,
    ubicacion_de_la_via TEXT, jerarquia_de_la_via TEXT, fecha DATE, hora TEXT,
    dia_1 TEXT, mes_1 TEXT, feriado TEXT, codigo_causa TEXT, causa_probable TEXT,
    tipo_de_siniestro TEXT, automovil DOUBLE PRECISION, bicicleta DOUBLE PRECISION,
    bus DOUBLE PRECISION, camion DOUBLE PRECISION, camioneta DOUBLE PRECISION,
    emergencias DOUBLE PRECISION, especial DOUBLE PRECISION, furgoneta DOUBLE PRECISION,
    motocicleta DOUBLE PRECISION, no_identificado DOUBLE PRECISION,
    scooter_electrico DOUBLE PRECISION, tricimoto DOUBLE PRECISION,
    vehiculo_deportivo_utilitario DOUBLE PRECISION, parroquia TEXT
);

CREATE TABLE IF NOT EXISTS public."Dataset_Limpio" (
    id_siniestro TEXT PRIMARY KEY, anio INT, fecha DATE, hora TIME,
    fecha_hora TIMESTAMPTZ, dia_semana TEXT, mes TEXT, feriado BOOLEAN,
    lesionados INT, fallecidos INT, latitud DOUBLE PRECISION, longitud DOUBLE PRECISION,
    direccion TEXT, nombre_via TEXT, ubicacion_via TEXT, jerarquia_via TEXT,
    codigo_causa TEXT, causa_probable TEXT, tipo_siniestro TEXT,
    automovil INT, bicicleta INT, bus INT, camion INT, camioneta INT,
    emergencias INT, especial INT, furgoneta INT, motocicleta INT,
    no_identificado INT, scooter_electrico INT, tricimoto INT,
    vehiculo_deportivo_utilitario INT, total_vehiculos INT, parroquia TEXT,
    via_perimetral_literal BOOLEAN, vehiculos_incompletos BOOLEAN, filas_origen INT
);

CREATE TABLE IF NOT EXISTS public."datasetDim_H" (
    tramo TEXT PRIMARY KEY, pieza INT, longitud_m DOUBLE PRECISION,
    nodo_inicio TEXT, nodo_fin TEXT, lat_inicio DOUBLE PRECISION, lon_inicio DOUBLE PRECISION,
    lat_fin DOUBLE PRECISION, lon_fin DOUBLE PRECISION, lat_centro DOUBLE PRECISION,
    lon_centro DOUBLE PRECISION, geometria JSONB, n_siniestros INT, fallecidos INT,
    lesionados INT, parroquia TEXT, referencia TEXT, casos_velocidad INT,
    h_velocidad DOUBLE PRECISION, h_velocidad_n DOUBLE PRECISION,
    casos_interseccion INT, h_interseccion DOUBLE PRECISION, h_interseccion_n DOUBLE PRECISION,
    casos_geometria INT, h_geometria DOUBLE PRECISION, h_geometria_n DOUBLE PRECISION,
    casos_visibilidad INT, h_visibilidad DOUBLE PRECISION, h_visibilidad_n DOUBLE PRECISION,
    valor_h DOUBLE PRECISION, nivel_h TEXT, tipos_principales TEXT,
    causas_principales TEXT, descripcion_h TEXT
);

CREATE TABLE IF NOT EXISTS public."datasetDim_E" (
    tramo TEXT PRIMARY KEY, pieza INT, longitud_m DOUBLE PRECISION,
    nodo_inicio TEXT, nodo_fin TEXT, lat_inicio DOUBLE PRECISION, lon_inicio DOUBLE PRECISION,
    lat_fin DOUBLE PRECISION, lon_fin DOUBLE PRECISION, lat_centro DOUBLE PRECISION,
    lon_centro DOUBLE PRECISION, geometria JSONB, n_siniestros INT, fallecidos INT,
    lesionados INT, parroquia TEXT, referencia TEXT, vehiculos INT,
    casos_hora_pico INT, casos_peatonal INT, e_vehiculos_km DOUBLE PRECISION,
    e_hora_pico DOUBLE PRECISION, e_peatonal DOUBLE PRECISION,
    e_vehiculos_km_n DOUBLE PRECISION, e_hora_pico_n DOUBLE PRECISION,
    e_peatonal_n DOUBLE PRECISION, valor_e DOUBLE PRECISION, nivel_e TEXT,
    hora_mas_frecuente INT, dia_mas_frecuente TEXT, vehiculo_mas_frecuente TEXT,
    descripcion_e TEXT
);

CREATE TABLE IF NOT EXISTS public."datasetDim_V" (
    tramo TEXT PRIMARY KEY, pieza INT, longitud_m DOUBLE PRECISION,
    nodo_inicio TEXT, nodo_fin TEXT, lat_inicio DOUBLE PRECISION, lon_inicio DOUBLE PRECISION,
    lat_fin DOUBLE PRECISION, lon_fin DOUBLE PRECISION, lat_centro DOUBLE PRECISION,
    lon_centro DOUBLE PRECISION, geometria JSONB, n_siniestros INT, fallecidos INT,
    lesionados INT, parroquia TEXT, referencia TEXT, casos_cruce_inseguro INT,
    casos_usuario_vulnerable INT, v_letalidad DOUBLE PRECISION, v_lesividad DOUBLE PRECISION,
    v_cruce_inseguro DOUBLE PRECISION, v_usuario_vulnerable DOUBLE PRECISION,
    v_letalidad_n DOUBLE PRECISION, v_lesividad_n DOUBLE PRECISION,
    v_cruce_inseguro_n DOUBLE PRECISION, v_usuario_vulnerable_n DOUBLE PRECISION,
    valor_v DOUBLE PRECISION, nivel_v TEXT, atropellos INT, motos INT,
    bicicletas INT, descripcion_v TEXT
);

NOTIFY pgrst, 'reload schema';
"""

with engine.begin() as conn:
    for stmt in SQL_TABLAS.strip().split(";"):
        if stmt.strip():
            conn.execute(text(stmt))

print("✔ Las 5 tablas fueron creadas en Supabase correctamente.")

✔ Las 5 tablas fueron creadas en Supabase correctamente.


### Funciones de acceso a Supabase (lectura paginada, escritura por lotes)

In [7]:
def leer(tabla, orden):
    """Lee una tabla completa de Supabase, de 1000 en 1000."""
    filas, desde = [], 0
    while True:
        r = sb_lectura.table(tabla).select('*').order(orden).range(desde, desde + 999).execute().data
        filas += r
        if len(r) < 1000:
            return pd.DataFrame(filas)
        desde += 1000

def contar(tabla, col):
    return sb_lectura.table(tabla).select(col, count='exact').limit(1).execute().count or 0

def a_registros(df):
    """DataFrame → lista de dicts JSON (NaN → null, numpy → Python, fechas → ISO)."""
    out = df.copy()
    for c in out.columns:
        if pd.api.types.is_datetime64_any_dtype(out[c]):
            out[c] = out[c].map(lambda x: x.isoformat() if pd.notna(x) else None)
    out = out.astype(object).where(pd.notna(out), None)
    conv = lambda o: o.item() if hasattr(o, 'item') else str(o)
    return json.loads(json.dumps(out.to_dict('records'), default=conv))

def reemplazar(tabla, df, pk, borrar_filtro=None):
    """Borra el contenido de la tabla y sube el DataFrame por lotes (idempotente)."""
    if sb_escritura is None:
        raise RuntimeError('Falta SUPABASE_SECRET_KEY en Secrets de Colab: es necesaria para escribir en Supabase.')
    q = sb_escritura.table(tabla).delete()
    (borrar_filtro(q) if borrar_filtro else q.neq(pk, '__ninguno__')).execute()
    regs = a_registros(df)
    for i in range(0, len(regs), LOTE):
        for intento in range(3):
            try:
                sb_escritura.table(tabla).insert(regs[i:i + LOTE]).execute(); break
            except Exception as e:
                if intento == 2: raise
                time.sleep(2 * (intento + 1))
    print(f'✔ {tabla}: {len(regs)} filas guardadas en Supabase')

## 1. Subir el Excel crudo a Supabase (`dataset_perimetral_raw`)
Se ejecuta **una sola vez**. Si la tabla ya tiene datos, se salta, salvo que pongas `FORZAR_RECARGA = True`. Sube las dos hojas (`Via_Perimetral` y `Todas_Variantes_Perimetral`) con una columna `hoja_origen` que indica de cuál viene cada fila.

In [13]:
HOJAS = ['Via_Perimetral', 'Todas_Variantes_Perimetral']
n_raw = contar('dataset_perimetral_raw', 'id_fila')
if n_raw > 0 and not FORZAR_RECARGA:
    print(f'dataset_perimetral_raw ya tiene {n_raw} filas en Supabase → no se vuelve a subir.')
else:
    if not os.path.exists(RUTA_EXCEL):
        from google.colab import files
        print(f'Selecciona el archivo {RUTA_EXCEL}…')
        RUTA_EXCEL = list(files.upload().keys())[0]
    partes = []
    for h in HOJAS:
        x = pd.read_excel(RUTA_EXCEL, sheet_name=h)
        x.columns = [c.strip().lower() for c in x.columns]
        x = x.rename(columns={'id': 'id_excel'})
        x['hoja_origen'] = h
        partes.append(x)
    crudo = pd.concat(partes, ignore_index=True)
    crudo['fecha'] = pd.to_datetime(crudo['fecha'], errors='coerce').dt.date
    crudo['hora'] = crudo['hora'].map(lambda h: None if pd.isna(h) else str(h))
    for c in ['id_excel', 'anio', 'lesionados', 'fallecidos']:
        crudo[c] = pd.to_numeric(crudo.get(c), errors='coerce').astype('Int64')
    cols = ['hoja_origen', 'id_excel', 'anio', 'siniestros', 'lesionados', 'fallecidos', 'latitud_y', 'longitud_x',
            'direccion', 'nombre_de_la_via', 'ubicacion_de_la_via', 'jerarquia_de_la_via', 'fecha', 'hora', 'dia_1',
            'mes_1', 'feriado', 'codigo_causa', 'causa_probable', 'tipo_de_siniestro'] + VEH + ['parroquia']
    crudo = crudo.reindex(columns=cols)
    reemplazar('dataset_perimetral_raw', crudo, 'id_fila', lambda q: q.gte('id_fila', 0))

✔ dataset_perimetral_raw: 5380 filas guardadas en Supabase


## 2. Traer el dataset crudo desde Supabase

In [16]:
raw = leer('dataset_perimetral_raw', 'id_fila')
print(f'{len(raw)} filas crudas ·', raw['hoja_origen'].value_counts().to_dict())
raw.head()

5380 filas crudas · {'Todas_Variantes_Perimetral': 3229, 'Via_Perimetral': 2151}


,id_fila,hoja_origen,id_excel,anio,siniestros,lesionados,fallecidos,latitud_y,longitud_x,direccion,...,camioneta,emergencias,especial,furgoneta,motocicleta,no_identificado,scooter_electrico,tricimoto,vehiculo_deportivo_utilitario,parroquia
0,1,Via_Perimetral,1.0,2024,ATM00001012024,0,0,-2.154674,-79.947396,"Via Perimetral, altura de la Urbanización Las ...",...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TARQUI
1,2,Via_Perimetral,2.0,2024,ATM00002012024,1,0,-2.234986,-79.935645,Vía Perimetral y 3° Pasaje 34 (antes de llegar...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,XIMENA
2,3,Via_Perimetral,3.0,2024,ATM00050012024,0,0,-2.186928,-79.958103,Oreja Vial que conecta la Via Perimetral con v...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TARQUI
3,4,Via_Perimetral,4.0,2024,ATM00086012024,1,0,-2.107297,-79.947697,Vía Perimetral y Calle 23C NO.,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,PASCUALES
4,5,Via_Perimetral,5.0,2024,ATM00087012024,2,0,-2.077257,-79.922544,Vía Perimetral (a 100 metros aproximadamente d...,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,PASCUALES


## 3. Limpieza y normalización → `Dataset_Limpio`
1. **Textos:** se recortan espacios, se unifican mayúsculas y se convierten `ND`, `N/D` y celdas vacías en faltantes.
2. **Columnas de vehículos desplazadas:** en algunas filas `AUTOMOVIL` viene vacío y los demás vehículos están corridos una columna a la derecha. Se regresan a su lugar y se marca `vehiculos_incompletos`.
3. **Duplicados:** se deja un registro por `SINIESTROS`, eligiendo la fila más completa. `filas_origen` dice cuántas filas crudas se unieron.
4. **Faltantes:**
   - vehículos → 0
   - feriado → el valor más frecuente del siniestro
   - causa → la descripción de su código
   - vía → `NO DISPONIBLE`
   - parroquia → `SIN DATO`
5. **Validaciones:** se eliminan coordenadas fuera de Guayaquil y horas o fechas inválidas.
6. **Campos derivados:** fecha y hora con zona horaria, día de la semana, mes y total de vehículos.

In [17]:
df = raw.copy()
log = {'filas_crudas': len(df)}

# 1) textos
TXT = ['siniestros', 'direccion', 'nombre_de_la_via', 'ubicacion_de_la_via', 'jerarquia_de_la_via', 'hora', 'dia_1',
       'mes_1', 'feriado', 'codigo_causa', 'causa_probable', 'tipo_de_siniestro', 'parroquia']
for c in TXT:
    df[c] = df[c].astype('string').str.strip().str.replace(r'\s+', ' ', regex=True)
for c in TXT[2:5] + TXT[6:]:
    df[c] = df[c].str.upper()
VACIOS = {'', 'ND', 'N/D', 'NAN', 'NONE', 'NULL', 'S/N'}
for c in TXT:
    if c not in ('direccion', 'siniestros'):
        df.loc[df[c].isin(VACIOS), c] = pd.NA
df = df[df['siniestros'].notna()]

# 2) vehículos desplazados
df[VEH] = df[VEH].apply(pd.to_numeric, errors='coerce')
sin_veh = df[VEH].isna().all(axis=1)
desplazada = df['automovil'].isna() & ~sin_veh
df.loc[desplazada, VEH] = df.loc[desplazada, VEH].shift(-1, axis=1).values
log['filas_vehiculos_desplazados_corregidas'] = int(desplazada.sum())
log['filas_sin_datos_de_vehiculos'] = int(sin_veh.sum())

# 3) duplicados: una fila por siniestro (la más completa)
df['_calidad'] = (~sin_veh).astype(int) * 2 + (~desplazada).astype(int)
df['_desplazada'] = desplazada
agg = df.groupby('siniestros').agg(
    filas_origen=('id_fila', 'size'),
    via_perimetral_literal=('hoja_origen', lambda s: (s == 'Via_Perimetral').any()),
    feriado_moda=('feriado', lambda s: s.mode().iat[0] if s.notna().any() else pd.NA))
limpio = (df.sort_values(['siniestros', '_calidad'], ascending=[True, False])
            .drop_duplicates('siniestros').set_index('siniestros'))
limpio = limpio.join(agg)
log['duplicados_eliminados'] = int(len(df) - len(limpio))

# 4) faltantes
limpio[VEH] = limpio[VEH].fillna(0).astype(int)
limpio['vehiculos_incompletos'] = limpio['_desplazada'] | limpio[VEH].sum(axis=1).eq(0)
limpio[['lesionados', 'fallecidos']] = limpio[['lesionados', 'fallecidos']].apply(pd.to_numeric, errors='coerce').fillna(0).astype(int)
causa_por_codigo = df.dropna(subset=['codigo_causa', 'causa_probable']).groupby('codigo_causa')['causa_probable'].agg(lambda s: s.mode().iat[0])
falt = {c: int(limpio[c].isna().sum()) for c in ['feriado_moda', 'causa_probable', 'nombre_de_la_via', 'parroquia', 'tipo_de_siniestro']}
limpio['causa_probable'] = limpio['causa_probable'].fillna(limpio['codigo_causa'].map(causa_por_codigo))
for c in ['nombre_de_la_via', 'ubicacion_de_la_via', 'jerarquia_de_la_via']:
    limpio[c] = limpio[c].fillna('NO DISPONIBLE')
limpio['parroquia'] = limpio['parroquia'].fillna('SIN DATO')
limpio['tipo_de_siniestro'] = limpio['tipo_de_siniestro'].fillna('NO ESPECIFICADO')
log['faltantes_normalizados'] = falt

# 5) validaciones
lat = pd.to_numeric(limpio['latitud_y'], errors='coerce'); lon = pd.to_numeric(limpio['longitud_x'], errors='coerce')
ok_xy = lat.between(BBOX_GYE[0], BBOX_GYE[2]) & lon.between(BBOX_GYE[1], BBOX_GYE[3])
hora = pd.to_datetime(limpio['hora'].str.slice(0, 8), format='%H:%M:%S', errors='coerce')
fecha = pd.to_datetime(limpio['fecha'], errors='coerce')
validos = ok_xy & hora.notna() & fecha.notna()
log['eliminados_por_coordenadas_invalidas'] = int((~ok_xy).sum())
log['eliminados_por_fecha_u_hora_invalida'] = int((ok_xy & ~(hora.notna() & fecha.notna())).sum())
limpio, lat, lon, hora, fecha = limpio[validos], lat[validos], lon[validos], hora[validos], fecha[validos]

# 6) derivados
DIAS = ['LUNES', 'MARTES', 'MIÉRCOLES', 'JUEVES', 'VIERNES', 'SÁBADO', 'DOMINGO']
MESES = ['ENERO', 'FEBRERO', 'MARZO', 'ABRIL', 'MAYO', 'JUNIO', 'JULIO', 'AGOSTO', 'SEPTIEMBRE', 'OCTUBRE', 'NOVIEMBRE', 'DICIEMBRE']
fh = (fecha + pd.to_timedelta(hora.dt.strftime('%H:%M:%S'))).dt.tz_localize('America/Guayaquil')

Dataset_Limpio = pd.DataFrame({
    'id_siniestro': limpio.index,
    'anio': fecha.dt.year.values, 'fecha': fecha.dt.date.values, 'hora': hora.dt.time.values, 'fecha_hora': fh.values,
    'dia_semana': fecha.dt.dayofweek.map(lambda i: DIAS[i]).values, 'mes': fecha.dt.month.map(lambda m: MESES[m - 1]).values,
    'feriado': limpio['feriado_moda'].eq('SI').values,
    'lesionados': limpio['lesionados'].values, 'fallecidos': limpio['fallecidos'].values,
    'latitud': lat.round(6).values, 'longitud': lon.round(6).values,
    'direccion': limpio['direccion'].values, 'nombre_via': limpio['nombre_de_la_via'].values,
    'ubicacion_via': limpio['ubicacion_de_la_via'].values, 'jerarquia_via': limpio['jerarquia_de_la_via'].values,
    'codigo_causa': limpio['codigo_causa'].values, 'causa_probable': limpio['causa_probable'].values,
    'tipo_siniestro': limpio['tipo_de_siniestro'].values,
    **{v: limpio[v].values for v in VEH},
    'total_vehiculos': limpio[VEH].sum(axis=1).values,
    'parroquia': limpio['parroquia'].values,
    'via_perimetral_literal': limpio['via_perimetral_literal'].astype(bool).values,
    'vehiculos_incompletos': limpio['vehiculos_incompletos'].astype(bool).values,
    'filas_origen': limpio['filas_origen'].astype(int).values,
}).sort_values('fecha_hora').reset_index(drop=True)
Dataset_Limpio['fecha_hora'] = pd.to_datetime(Dataset_Limpio['fecha_hora']).dt.tz_localize('UTC')   # instante exacto, se guarda con +00:00
log['filas_finales'] = len(Dataset_Limpio)

print('Reporte de limpieza:')
for k, v in log.items():
    print(f'  {k}: {v}')
Dataset_Limpio.head()

Reporte de limpieza:
  filas_crudas: 5380
  filas_vehiculos_desplazados_corregidas: 590
  filas_sin_datos_de_vehiculos: 500
  duplicados_eliminados: 4504
  faltantes_normalizados: {'feriado_moda': 0, 'causa_probable': 0, 'nombre_de_la_via': 399, 'parroquia': 0, 'tipo_de_siniestro': 0}
  eliminados_por_coordenadas_invalidas: 0
  eliminados_por_fecha_u_hora_invalida: 0
  filas_finales: 876


C:\Users\josue\AppData\Local\Temp\ipykernel_11868\728908659.py:68: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  'dia_semana': fecha.dt.dayofweek.map(lambda i: DIAS[i]).values, 'mes': fecha.dt.month.map(lambda m: MESES[m - 1]).values,


,id_siniestro,anio,fecha,hora,fecha_hora,dia_semana,mes,feriado,lesionados,fallecidos,...,motocicleta,no_identificado,scooter_electrico,tricimoto,vehiculo_deportivo_utilitario,total_vehiculos,parroquia,via_perimetral_literal,vehiculos_incompletos,filas_origen
0,ATM00001012024,2024,2024-01-01,02:01:59,2024-01-01 07:01:59+00:00,LUNES,ENERO,True,0,0,...,0,0,0,0,0,1,TARQUI,True,False,8
1,ATM00002012024,2024,2024-01-01,03:01:59,2024-01-01 08:01:59+00:00,LUNES,ENERO,True,1,0,...,0,0,0,0,0,1,XIMENA,True,False,8
2,ATM00025012024,2024,2024-01-02,02:01:59,2024-01-02 07:01:59+00:00,MARTES,ENERO,False,2,0,...,1,0,0,0,0,1,XIMENA,False,False,4
3,ATM00050012024,2024,2024-01-05,03:01:59,2024-01-05 08:01:59+00:00,VIERNES,ENERO,False,0,0,...,0,0,0,0,0,1,TARQUI,True,False,8
4,ATM00057012024,2024,2024-01-06,10:01:59,2024-01-06 15:01:59+00:00,SÁBADO,ENERO,False,0,1,...,0,0,0,0,0,2,PASCUALES,False,False,4


## 4. Subir `Dataset_Limpio` a Supabase

In [18]:
reemplazar('Dataset_Limpio', Dataset_Limpio, 'id_siniestro')

✔ Dataset_Limpio: 876 filas guardadas en Supabase


In [30]:
import os, ssl, json
import pandas as pd
from urllib.parse import urlsplit, urlunsplit
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine, text
from supabase import create_client

load_dotenv(find_dotenv(), override=True)

# 1. Crear la tabla Dataset_Limpio_Censo con lectura pública (RLS)
parts = urlsplit(os.getenv("DATABASE_URL"))
db_url = urlunsplit(("postgresql+pg8000", parts.netloc, parts.path, "", ""))
ssl_ctx = ssl.create_default_context()
ssl_ctx.check_hostname = False
ssl_ctx.verify_mode = ssl.CERT_NONE
engine = create_engine(db_url, connect_args={"ssl_context": ssl_ctx})

SQL_CENSO = """
CREATE TABLE IF NOT EXISTS public."Dataset_Limpio_Censo" (
    id_siniestro TEXT PRIMARY KEY, anio INT, fecha DATE, hora TIME,
    fecha_hora TIMESTAMPTZ, dia_semana TEXT, mes TEXT, feriado BOOLEAN,
    lesionados INT, fallecidos INT, latitud DOUBLE PRECISION, longitud DOUBLE PRECISION,
    direccion TEXT, nombre_via TEXT, ubicacion_via TEXT, jerarquia_via TEXT,
    codigo_causa TEXT, causa_probable TEXT, tipo_siniestro TEXT,
    automovil INT, bicicleta INT, bus INT, camion INT, camioneta INT,
    emergencias INT, especial INT, furgoneta INT, motocicleta INT,
    no_identificado INT, scooter_electrico INT, tricimoto INT,
    vehiculo_deportivo_utilitario INT, total_vehiculos INT, parroquia TEXT,
    via_perimetral_literal BOOLEAN, vehiculos_incompletos BOOLEAN, filas_origen INT,
    cod_sector BIGINT, sector_poblacion INT, sector_viviendas INT,
    sector_pct_65mas DOUBLE PRECISION, sector_pct_menor15 DOUBLE PRECISION,
    sector_pct_discapacidad_grave DOUBLE PRECISION, sector_pct_dificultad_caminar DOUBLE PRECISION,
    sector_pct_pobreza_nbi DOUBLE PRECISION, sector_pct_sin_instruccion_15mas DOUBLE PRECISION,
    dist_sector_m DOUBLE PRECISION, densidad_sector_hab_km2 DOUBLE PRECISION,
    entorno300_sectores INT, entorno300_poblacion INT, entorno300_viviendas INT,
    entorno300_pct_65mas DOUBLE PRECISION, entorno300_pct_menor15 DOUBLE PRECISION,
    entorno300_pct_discapacidad_grave DOUBLE PRECISION, entorno300_pct_dificultad_caminar DOUBLE PRECISION,
    entorno300_pct_pobreza_nbi DOUBLE PRECISION, entorno300_pct_sin_instruccion_15mas DOUBLE PRECISION,
    entorno300_densidad_hab_km2 DOUBLE PRECISION,
    cod_manzana BIGINT, dist_manzana_m DOUBLE PRECISION,
    manzana_edificios INT, manzana_poblacion_est DOUBLE PRECISION,
    manzana_equipamientos INT, manzana_educacion INT
);
GRANT SELECT ON TABLE public."Dataset_Limpio_Censo" TO anon, authenticated;
ALTER TABLE public."Dataset_Limpio_Censo" ENABLE ROW LEVEL SECURITY;
DROP POLICY IF EXISTS "Lectura publica" ON public."Dataset_Limpio_Censo";
CREATE POLICY "Lectura publica" ON public."Dataset_Limpio_Censo" FOR SELECT USING (true);
NOTIFY pgrst, 'reload schema';
"""

with engine.begin() as conn:
    for stmt in SQL_CENSO.strip().split(";"):
        if stmt.strip():
            conn.execute(text(stmt))

# 2. Leer Perimetral_24_26_con_Censo.xlsx y subir las 876 filas
df_c = pd.read_excel("Perimetral_24_26_con_Censo.xlsx", sheet_name="Siniestros_Censo")
df_c['fecha'] = pd.to_datetime(df_c['fecha']).dt.date.astype(str)
df_c['hora'] = df_c['hora'].astype(str)
df_c['fecha_hora'] = (pd.to_datetime(df_c['fecha_hora'])
                        .dt.tz_localize('America/Guayaquil')
                        .dt.tz_convert('UTC')
                        .map(lambda x: x.isoformat()))

for col in ['cod_sector', 'cod_manzana', 'manzana_edificios', 'manzana_equipamientos', 'manzana_educacion']:
    df_c[col] = pd.to_numeric(df_c[col], errors='coerce').astype('Int64')

out = df_c.astype(object).where(pd.notna(df_c), None)
regs = json.loads(json.dumps(out.to_dict('records'), default=lambda o: o.item() if hasattr(o, 'item') else str(o)))

sb_esc = create_client(os.getenv("SUPABASE_URL"), os.getenv("SUPABASE_KEY"))
sb_esc.table("Dataset_Limpio_Censo").delete().neq("id_siniestro", "__ninguno__").execute()
for i in range(0, len(regs), 500):
    sb_esc.table("Dataset_Limpio_Censo").insert(regs[i:i + 500]).execute()

print(f"✔ Dataset_Limpio_Censo: {len(regs)} filas subidas a Supabase.")

✔ Dataset_Limpio_Censo: 876 filas subidas a Supabase.


---
---
&nbsp;

# ⏸️ ESPACIO

&nbsp;

A partir de aquí todo se calcula **leyendo `Dataset_Limpio` desde Supabase**. Esta segunda parte se puede ejecutar sola, sin repetir la limpieza: basta con correr antes la sección 0.

---
---

## 5. Traer `Dataset_Limpio` desde Supabase

In [19]:
DL = leer('Dataset_Limpio', 'id_siniestro')
DL['fecha_hora'] = pd.to_datetime(DL['fecha_hora'], utc=True).dt.tz_convert('America/Guayaquil')
DL['hora_num'] = DL['fecha_hora'].dt.hour
gdl = gpd.GeoDataFrame(DL, geometry=gpd.points_from_xy(DL['longitud'], DL['latitud']), crs=4326).to_crs(CRS_METROS)
print(f'{len(gdl)} siniestros limpios cargados desde Supabase')

876 siniestros limpios cargados desde Supabase


## 6. Red vial: tramos de 500 m (aristas) y nodos
El trazado de la Perimetral se descarga de OpenStreetMap y se guarda en caché (`eje_perimetral.geojson`). Después:
- La vía se corta en tramos de unos 500 m. Cada tramo es una **arista**.
- Sus extremos son los **nodos**. Los extremos que están a menos de 30 m entre sí se consideran el mismo nodo.
- Cada siniestro se asigna al tramo más cercano, hasta un máximo de 250 m.

In [20]:
import requests, time
from shapely.geometry import LineString
from shapely.ops import linemerge, unary_union, substring

BBOX = '(-2.27,-79.99,-2.05,-79.86)'   # sur, oeste, norte, este
HW = '^(motorway|trunk|primary|secondary|tertiary)$'
QUERY = f'''[out:json][timeout:120];
(
  way["highway"~"{HW}"]["name"~"Perimetral",i]{BBOX};
  way["highway"~"{HW}"]["alt_name"~"Perimetral",i]{BBOX};
  way["highway"~"{HW}"]["official_name"~"Perimetral",i]{BBOX};
);
out geom;'''
SERVIDORES = ['https://overpass-api.de/api/interpreter',
              'https://overpass.kumi.systems/api/interpreter',
              'https://maps.mail.ru/osm/tools/overpass/api/interpreter']

def descargar_osm():
    for url in SERVIDORES:
        try:
            r = requests.post(url, data={'data': QUERY}, timeout=180,
                              headers={'User-Agent': 'RISKTWIN-notebook/1.0'})
            r.raise_for_status()
            els = r.json().get('elements', [])
            if els:
                print('OSM:', url, '→', len(els), 'vías')
                return els
        except Exception as e:
            print('Falló', url, '→', e); time.sleep(2)
    raise RuntimeError('No se pudo descargar la Perimetral desde Overpass. Reintenta en unos minutos.')

CACHE_EJE = 'eje_perimetral.geojson'
if os.path.exists(CACHE_EJE):
    eje = list(gpd.read_file(CACHE_EJE).to_crs(CRS_METROS).geometry)
    print(f'Eje leído de la caché: {len(eje)} piezas')
else:
    ways = descargar_osm()
    lineas = [LineString([(p['lon'], p['lat']) for p in w['geometry']])
              for w in ways if w.get('geometry') and len(w['geometry']) > 1]
    lineas_m = list(gpd.GeoSeries(lineas, crs=4326).to_crs(CRS_METROS))

    fusion = linemerge(unary_union(lineas_m))
    cadenas = sorted(getattr(fusion, 'geoms', [fusion]), key=lambda g: g.length, reverse=True)

    CONEXION_M, SOLAPE_M = 150, 60
    eje, pendientes = [cadenas[0]], cadenas[1:]
    cambio = True
    while cambio:
        cambio = False
        actual = unary_union(eje)
        zona_ocupada = actual.buffer(SOLAPE_M)
        for c in list(pendientes):
            solape = c.intersection(zona_ocupada).length / c.length
            if solape >= 0.5 or c.length < 100:
                pendientes.remove(c)
            elif c.distance(actual) <= CONEXION_M:
                resto = c.difference(zona_ocupada)
                eje += [p for p in getattr(resto, 'geoms', [resto]) if p.length >= 100]
                pendientes.remove(c); cambio = True

    print(f'Eje de la Perimetral: {len(eje)} piezas, {sum(g.length for g in eje)/1000:.1f} km')
    gpd.GeoDataFrame(geometry=eje, crs=CRS_METROS).to_crs(4326).to_file(CACHE_EJE, driver='GeoJSON')

OSM: https://overpass-api.de/api/interpreter → 149 vías
Eje de la Perimetral: 18 piezas, 43.0 km


In [21]:
from shapely.geometry import Point, mapping
from shapely.ops import substring

# aristas (tramos)
filas = []
for p, pieza in enumerate(eje, 1):
    n = max(1, round(pieza.length / LONGITUD_TRAMO_M))
    paso = pieza.length / n
    for i in range(n):
        filas.append({'pieza': p, 'geometry': substring(pieza, i * paso, (i + 1) * paso)})
tramos = gpd.GeoDataFrame(filas, crs=CRS_METROS)
orden = tramos.assign(_y=tramos.centroid.y).groupby('pieza')['_y'].max().sort_values(ascending=False).index
tramos['pieza'] = pd.Categorical(tramos['pieza'], categories=orden, ordered=True)
tramos = tramos.sort_values('pieza', kind='stable').reset_index(drop=True)
tramos['pieza'] = tramos['pieza'].astype(int)
tramos['tramo'] = [f'T{i:03d}' for i in range(1, len(tramos) + 1)]
tramos['longitud_m'] = tramos.length.round(1)

# nodos: extremos de cada tramo, fusionando los cercanos
nodos_xy, ids_ini, ids_fin = [], [], []
def nodo_de(pt):
    for k, q in enumerate(nodos_xy):
        if pt.distance(q) <= TOLERANCIA_NODO_M:
            return f'N{k + 1:03d}'
    nodos_xy.append(pt); return f'N{len(nodos_xy):03d}'
for g in tramos.geometry:
    ids_ini.append(nodo_de(Point(g.coords[0]))); ids_fin.append(nodo_de(Point(g.coords[-1])))
tramos['nodo_inicio'], tramos['nodo_fin'] = ids_ini, ids_fin
nodos = gpd.GeoDataFrame({'nodo': [f'N{k + 1:03d}' for k in range(len(nodos_xy))]}, geometry=nodos_xy, crs=CRS_METROS)
grado = pd.concat([tramos['nodo_inicio'], tramos['nodo_fin']]).value_counts()
nodos['grado'] = nodos['nodo'].map(grado).fillna(0).astype(int)

# coordenadas en WGS84 para Supabase y mapas
t4 = tramos.to_crs(4326)
tramos['lat_inicio'] = [g.coords[0][1] for g in t4.geometry];  tramos['lon_inicio'] = [g.coords[0][0] for g in t4.geometry]
tramos['lat_fin']    = [g.coords[-1][1] for g in t4.geometry]; tramos['lon_fin']    = [g.coords[-1][0] for g in t4.geometry]
cen = t4.geometry.interpolate(0.5, normalized=True)
tramos['lat_centro'], tramos['lon_centro'] = cen.y.values, cen.x.values
tramos['geometria'] = [json.loads(json.dumps(mapping(g))) for g in t4.geometry]

# asignación siniestro → tramo
asig = gpd.sjoin_nearest(gdl, tramos[['tramo', 'geometry']], how='left', max_distance=DISTANCIA_MAX_M, distance_col='dist_m')
gdl = asig.sort_values('dist_m').drop_duplicates('id_siniestro').drop(columns='index_right')
d = gdl[gdl['tramo'].notna()].copy()
print(f'{len(tramos)} tramos (aristas) · {len(nodos)} nodos · '
      f'{len(d)} de {len(gdl)} siniestros asignados ({len(d)/len(gdl):.0%})')

88 tramos (aristas) · 74 nodos · 816 de 876 siniestros asignados (93%)


C:\Users\josue\AppData\Local\Temp\ipykernel_11868\3247317981.py:37: UserWarning: Geometry is in a geographic CRS. Results from 'interpolate' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  cen = t4.geometry.interpolate(0.5, normalized=True)


In [22]:
# Utilidades comunes a las tres dimensiones
moda = lambda s: s.mode().iat[0] if not s.mode().empty else None
txt = lambda v, k=None: ('' if v is None or (isinstance(v, float) and np.isnan(v)) else str(v))[:k]
top = lambda s, k=3: '; '.join(f'{i} ({c})' for i, c in s.value_counts().head(k).items())

base = tramos[['tramo', 'pieza', 'longitud_m', 'nodo_inicio', 'nodo_fin', 'lat_inicio', 'lon_inicio', 'lat_fin',
               'lon_fin', 'lat_centro', 'lon_centro', 'geometria']].copy()
gen = d.groupby('tramo').agg(n_siniestros=('id_siniestro', 'size'), fallecidos=('fallecidos', 'sum'),
                             lesionados=('lesionados', 'sum'), parroquia=('parroquia', moda))
mid = tramos.set_index('tramo').geometry.interpolate(0.5, normalized=True)
ref = (d.assign(_dm=[g.distance(mid[t]) for g, t in zip(d.geometry, d['tramo'])])
         .sort_values('_dm').drop_duplicates('tramo').set_index('tramo')['direccion'].rename('referencia'))
base = base.merge(gen, on='tramo', how='left').merge(ref, on='tramo', how='left')
base[['n_siniestros', 'fallecidos', 'lesionados']] = base[['n_siniestros', 'fallecidos', 'lesionados']].fillna(0).astype(int)
base['parroquia'] = base['parroquia'].fillna('SIN DATO')
CON = base['n_siniestros'] > 0

def tasa(casos, p):                     # porcentaje suavizado hacia el promedio de la vía
    return (casos + SUAVIZADO * p) / (base['n_siniestros'] + SUAVIZADO)

def minmax(s):
    s = s.where(CON)
    lo, hi = s.min(), s.max()
    return ((s - lo) / (hi - lo) if hi > lo else s * 0).fillna(0.0)

def nivel(v, n):
    if n == 0: return 'Sin registros'
    return 'Baja' if v < UMBRALES[0] else ('Media' if v < UMBRALES[1] else 'Alta')

def conteo(flag):                       # nº de siniestros del tramo que cumplen la condición
    return base['tramo'].map(d[flag].groupby(d['tramo']).sum()).fillna(0).astype(int)

# ---- mapas ----
import folium, branca

# Mapa base CARTO con API key (evita la marca de agua "API KEY REQUIRED")
CARTO_KEY = 'cb1_5do9_1_90e38ab2e2043b7ab55c5b13'
TILES_CARTO = ('https://basemaps.cartocdn.com/rastertiles/light_all/'
               '{z}/{x}/{y}.png?key=' + CARTO_KEY)
ATTR_CARTO = '© OpenStreetMap contributors © CARTO'

def mapa_dimension(df_dim, col, titulo, campos, puntos=None, nombre_puntos='Siniestros'):
    cmap = branca.colormap.LinearColormap(['#1a9850', '#fee08b', '#d73027'], vmin=0, vmax=1, caption=titulo)
    m = folium.Map(location=[df_dim['lat_centro'].mean(), df_dim['lon_centro'].mean()],
                   zoom_start=13, tiles=TILES_CARTO, attr=ATTR_CARTO)
    fg = folium.FeatureGroup(name=f'Tramos · {titulo}')
    for r in df_dim.itertuples():
        color = '#9e9e9e' if r.n_siniestros == 0 else cmap(getattr(r, col))
        filas = ''.join(f'<tr><td style="padding-right:8px"><b>{et}</b></td><td>{fmt(getattr(r, c))}</td></tr>' for et, c, fmt in campos)
        html = f'<div style="font-family:sans-serif;font-size:12px;min-width:300px"><b style="font-size:14px">{r.tramo}</b> · {r.parroquia}<br><i>{txt(r.referencia, 110)}</i><table>{filas}</table></div>'
        folium.GeoJson(r.geometria, style_function=lambda f, c=color: {'color': c, 'weight': 7, 'opacity': .9},
                       highlight_function=lambda f: {'weight': 12, 'color': '#000'},
                       tooltip=f'{r.tramo} · {titulo}: {getattr(r, col):.2f}', popup=folium.Popup(html, max_width=420)).add_to(fg)
    fg.add_to(m)
    fn = folium.FeatureGroup(name='Nodos', show=True)
    for n in nodos.to_crs(4326).itertuples():
        folium.CircleMarker([n.geometry.y, n.geometry.x], radius=3 + n.grado, color='#253494', fill=True, fill_opacity=.9,
                            tooltip=f'Nodo {n.nodo} · grado {n.grado}').add_to(fn)
    fn.add_to(m)
    if puntos is not None and len(puntos):
        fp = folium.FeatureGroup(name=nombre_puntos, show=False)
        for s in puntos.to_crs(4326).itertuples():
            folium.CircleMarker([s.geometry.y, s.geometry.x], radius=3, color='#000', fill=True, fill_opacity=.7,
                                tooltip=f'{s.id_siniestro} · {s.tipo_siniestro} · {s.fecha_hora:%Y-%m-%d %H:%M}').add_to(fp)
        fp.add_to(m)
    cmap.add_to(m); folium.LayerControl(collapsed=False).add_to(m)
    return m

f2 = lambda v: f'{v:.2f}'
f0 = lambda v: f'{int(v)}'
fp_ = lambda v: f'{v:.0%}'
ft = lambda v: v if isinstance(v, str) else '—'
print('Listo para calcular las dimensiones')

Listo para calcular las dimensiones


## 7. Dimensión H — Peligrosidad → `datasetDim_H`
| Indicador | Condición en `Dataset_Limpio` |
|---|---|
| Velocidad | `codigo_causa` = C09 |
| Intersecciones conflictivas | `tipo_siniestro` = CHOQUE LATERAL / FRONTAL, o `codigo_causa` ∈ C18, C23, C25, C28 |
| Geometría vial | `tipo_siniestro` ∈ PÉRDIDA DE CARRIL / PISTA, VOLCAMIENTOS, ESTRELLAMIENTOS |
| Visibilidad | hora 18:00–05:59 o `codigo_causa` = C10 |

**Cálculo de cada tramo:**
1. Porcentaje suavizado: (casos + 5·p) ÷ (n + 5).
2. Normalización min-max entre tramos.
3. **H** = promedio de los 4 indicadores.

In [23]:
d['f_velocidad']    = d['codigo_causa'].eq('C09')
d['f_interseccion'] = d['tipo_siniestro'].isin(['CHOQUE LATERAL', 'CHOQUE FRONTAL']) | d['codigo_causa'].isin(['C18', 'C23', 'C25', 'C28'])
d['f_geometria']    = d['tipo_siniestro'].isin(['PÉRDIDA DE CARRIL', 'PÉRDIDA DE PISTA', 'VOLCAMIENTOS', 'ESTRELLAMIENTOS'])
d['f_visibilidad']  = d['codigo_causa'].eq('C10') | (d['hora_num'] >= 18) | (d['hora_num'] < 6)

H = base.copy()
for k in ['velocidad', 'interseccion', 'geometria', 'visibilidad']:
    H[f'casos_{k}'] = conteo(f'f_{k}')
    H[f'h_{k}'] = tasa(H[f'casos_{k}'], d[f'f_{k}'].mean())
    H[f'h_{k}_n'] = minmax(H[f'h_{k}'])
H['valor_h'] = H[[f'h_{k}_n' for k in ['velocidad', 'interseccion', 'geometria', 'visibilidad']]].mean(axis=1).where(CON, 0.0)
H['nivel_h'] = [nivel(v, n) for v, n in zip(H['valor_h'], H['n_siniestros'])]
H = H.merge(d.groupby('tramo').agg(tipos_principales=('tipo_siniestro', top),
                                   causas_principales=('causa_probable', lambda s: top(s.str.slice(0, 50)))), on='tramo', how='left')
def desc_h(r):
    if r.n_siniestros == 0: return 'Sin siniestros registrados en 2024-2026.'
    return (f'Peligrosidad {r.nivel_h.lower()} ({r.valor_h:.2f}). De {r.n_siniestros} siniestros: {r.casos_velocidad} por exceso de velocidad, '
            f'{r.casos_interseccion} en intersecciones conflictivas, {r.casos_geometria} por pérdida de carril o volcamiento '
            f'y {r.casos_visibilidad} de noche o con mal clima.')
H['descripcion_h'] = [desc_h(r) for r in H.itertuples()]
datasetDim_H = H
reemplazar('datasetDim_H', datasetDim_H, 'tramo')
datasetDim_H.drop(columns='geometria').sort_values('valor_h', ascending=False).head(10)

✔ datasetDim_H: 88 filas guardadas en Supabase


,tramo,pieza,longitud_m,nodo_inicio,nodo_fin,lat_inicio,lon_inicio,lat_fin,lon_fin,lat_centro,...,h_geometria,h_geometria_n,casos_visibilidad,h_visibilidad,h_visibilidad_n,valor_h,nivel_h,tipos_principales,causas_principales,descripcion_h
56,T057,1,508.2,N046,N047,-2.172345,-79.944948,-2.168163,-79.946796,-2.170188,...,0.586275,0.953571,3,0.554289,0.549520,0.646939,Media,PÉRDIDA DE CARRIL (3); VOLCAMIENTOS (1); CHOQU...,CONDUCIR VEHÍCULO SUPERANDO LOS LÍMITES MÁXIMO...,Peligrosidad media (0.65). De 5 siniestros: 5 ...
10,T011,11,491.5,N011,N012,-2.087705,-79.947217,-2.092011,-79.948053,-2.089794,...,0.607843,1.000000,3,0.692862,0.839545,0.642010,Media,ESTRELLAMIENTOS (3),CONDUCIR VEHÍCULO SUPERANDO LOS LÍMITES MÁXIMO...,Peligrosidad media (0.64). De 3 siniestros: 2 ...
19,T020,10,491.5,N014,N013,-2.100932,-79.947788,-2.096486,-79.947846,-2.098709,...,0.532977,0.838843,4,0.594808,0.634325,0.639147,Media,ESTRELLAMIENTOS (3); COLISIÓN (1); PÉRDIDA DE ...,CONDUCIR VEHÍCULO SUPERANDO LOS LÍMITES MÁXIMO...,Peligrosidad media (0.64). De 6 siniestros: 5 ...
43,T044,1,508.2,N033,N034,-2.225598,-79.956230,-2.221149,-79.957200,-2.223431,...,0.586275,0.953571,3,0.554289,0.549520,0.590608,Media,PÉRDIDA DE CARRIL (3); ROZAMIENTOS (1); ESTREL...,CONDUCIR VEHÍCULO SUPERANDO LOS LÍMITES MÁXIMO...,Peligrosidad media (0.59). De 5 siniestros: 4 ...
55,T056,1,508.2,N045,N046,-2.176828,-79.945504,-2.172345,-79.944948,-2.174626,...,0.540305,0.854618,2,0.504766,0.445870,0.579350,Media,PÉRDIDA DE CARRIL (3); CHOQUE LATERAL (1),CONDUCIR VEHÍCULO SUPERANDO LOS LÍMITES MÁXIMO...,Peligrosidad media (0.58). De 4 siniestros: 3 ...
30,T031,6,463.0,N021,N022,-2.114774,-79.947953,-2.118906,-79.948633,-2.116842,...,0.521338,0.813789,8,0.620170,0.687406,0.577739,Media,PÉRDIDA DE CARRIL (4); ESTRELLAMIENTOS (3); CH...,CONDUCIR VEHÍCULO SUPERANDO LOS LÍMITES MÁXIMO...,Peligrosidad media (0.58). De 12 siniestros: 9...
81,T082,2,478.4,N068,N069,-2.235190,-79.935763,-2.236563,-79.931685,-2.235873,...,0.429194,0.615440,3,0.615877,0.678420,0.577693,Media,ESTRELLAMIENTOS (2); CHOQUE LATERAL (1); ATROP...,CONDUCIR VEHÍCULO SUPERANDO LOS LÍMITES MÁXIMO...,Peligrosidad media (0.58). De 4 siniestros: 3 ...
32,T033,8,447.8,N023,N022,-2.123036,-79.949211,-2.119042,-79.948542,-2.121037,...,0.524183,0.819913,9,0.769526,1.000000,0.563211,Media,ESTRELLAMIENTOS (3); PÉRDIDA DE CARRIL (3); CH...,CONDUCIR VEHÍCULO SUPERANDO LOS LÍMITES MÁXIMO...,Peligrosidad media (0.56). De 10 siniestros: 5...
82,T083,2,478.4,N069,N070,-2.236563,-79.931685,-2.237934,-79.927605,-2.237248,...,0.488562,0.743236,2,0.378574,0.181758,0.548788,Media,PÉRDIDA DE CARRIL (4); CHOQUE LATERAL (2); ROZ...,CONDUCIR VEHÍCULO SUPERANDO LOS LÍMITES MÁXIMO...,Peligrosidad media (0.55). De 7 siniestros: 6 ...
17,T018,10,491.5,N016,N015,-2.109823,-79.947716,-2.105378,-79.947755,-2.107601,...,0.488562,0.743236,4,0.545241,0.530583,0.548603,Media,PÉRDIDA DE CARRIL (4); CHOQUE LATERAL (1); COL...,CONDUCIR VEHÍCULO SUPERANDO LOS LÍMITES MÁXIMO...,Peligrosidad media (0.55). De 7 siniestros: 5 ...


In [24]:
mapa_dimension(datasetDim_H, 'valor_h', 'Peligrosidad (H)', [
    ('H', 'valor_h', f2), ('Nivel', 'nivel_h', ft), ('Siniestros', 'n_siniestros', f0),
    ('Velocidad', 'casos_velocidad', f0), ('Intersecciones', 'casos_interseccion', f0),
    ('Geometría vial', 'casos_geometria', f0), ('Visibilidad', 'casos_visibilidad', f0),
    ('Nodos', 'nodo_inicio', ft), ('→', 'nodo_fin', ft), ('Descripción', 'descripcion_h', ft)],
    puntos=d[d['f_velocidad']], nombre_puntos='Siniestros por velocidad (C09)')

## 8. Dimensión E — Exposición → `datasetDim_E`
| Indicador | Condición en `Dataset_Limpio` |
|---|---|
| Tráfico vehicular | `total_vehiculos` ÷ km del tramo (es una cantidad, sin suavizado) |
| Horarios de mayor movilidad | hora 06–08 o 17–19 |
| Flujo peatonal | `tipo_siniestro` ∈ ATROPELLOS, ARROLLAMIENTOS, CAÍDA DE PASAJERO |

In [25]:
d['f_hora_pico'] = d['hora_num'].isin(HORAS_PICO)
d['f_peatonal']  = d['tipo_siniestro'].isin(['ATROPELLOS', 'ARROLLAMIENTOS', 'CAÍDA DE PASAJERO'])

E = base.copy()
E['vehiculos'] = E['tramo'].map(d.groupby('tramo')['total_vehiculos'].sum()).fillna(0).astype(int)
E['casos_hora_pico'] = conteo('f_hora_pico'); E['casos_peatonal'] = conteo('f_peatonal')
E['e_vehiculos_km'] = E['vehiculos'] / (E['longitud_m'] / 1000)
E['e_hora_pico'] = tasa(E['casos_hora_pico'], d['f_hora_pico'].mean())
E['e_peatonal'] = tasa(E['casos_peatonal'], d['f_peatonal'].mean())
for c in ['e_vehiculos_km', 'e_hora_pico', 'e_peatonal']:
    E[c + '_n'] = minmax(E[c])
E['valor_e'] = E[['e_vehiculos_km_n', 'e_hora_pico_n', 'e_peatonal_n']].mean(axis=1).where(CON, 0.0)
E['nivel_e'] = [nivel(v, n) for v, n in zip(E['valor_e'], E['n_siniestros'])]
vehs = d.groupby('tramo')[VEH].sum()
extra = pd.DataFrame({'hora_mas_frecuente': d.groupby('tramo')['hora_num'].agg(moda),
                      'dia_mas_frecuente': d.groupby('tramo')['dia_semana'].agg(moda),
                      'vehiculo_mas_frecuente': vehs.idxmax(axis=1).where(vehs.sum(axis=1) > 0).str.upper()})
E = E.merge(extra, on='tramo', how='left')
E['hora_mas_frecuente'] = E['hora_mas_frecuente'].astype('Int64')
def desc_e(r):
    if r.n_siniestros == 0: return 'Sin siniestros registrados en 2024-2026.'
    return (f'Exposición {r.nivel_e.lower()} ({r.valor_e:.2f}). {r.vehiculos} vehículos involucrados '
            f'({r.e_vehiculos_km:.0f}/km), {r.casos_hora_pico} siniestros en hora pico y {r.casos_peatonal} con peatones. '
            f'Hora más frecuente: {int(r.hora_mas_frecuente)}:00; día: {r.dia_mas_frecuente}.')
E['descripcion_e'] = [desc_e(r) for r in E.itertuples()]
datasetDim_E = E
reemplazar('datasetDim_E', datasetDim_E, 'tramo')
datasetDim_E.drop(columns='geometria').sort_values('valor_e', ascending=False).head(10)

✔ datasetDim_E: 88 filas guardadas en Supabase


,tramo,pieza,longitud_m,nodo_inicio,nodo_fin,lat_inicio,lon_inicio,lat_fin,lon_fin,lat_centro,...,e_peatonal,e_vehiculos_km_n,e_hora_pico_n,e_peatonal_n,valor_e,nivel_e,hora_mas_frecuente,dia_mas_frecuente,vehiculo_mas_frecuente,descripcion_e
34,T035,7,449.1,N023,N025,-2.123065,-79.949104,-2.119077,-79.948342,-2.121079,...,0.525210,0.415617,0.445833,1.000000,0.620483,Media,7,LUNES,MOTOCICLETA,Exposición media (0.62). 20 vehículos involucr...
64,T065,1,508.2,N054,N055,-2.136901,-79.946536,-2.132528,-79.947945,-2.134716,...,0.135930,0.942270,0.653635,0.178282,0.591396,Media,6,DOMINGO,MOTOCICLETA,Exposición media (0.59). 50 vehículos involucr...
85,T086,2,478.4,N072,N063,-2.240247,-79.919350,-2.239673,-79.915121,-2.240176,...,0.502941,0.082313,0.728775,0.952993,0.588027,Media,1,DOMINGO,MOTOCICLETA,Exposición media (0.59). 5 vehículos involucra...
28,T029,12,608.5,N018,N019,-2.103323,-79.948297,-2.108804,-79.947993,-2.106053,...,0.366310,0.092603,1.000000,0.664584,0.585729,Media,8,MIÉRCOLES,NO_IDENTIFICADO,Exposición media (0.59). 7 vehículos involucra...
83,T084,2,478.4,N070,N071,-2.237934,-79.927605,-2.239338,-79.923537,-2.238637,...,0.463801,0.204965,0.671885,0.870374,0.582408,Media,7,LUNES,NO_IDENTIFICADO,Exposición media (0.58). 11 vehículos involucr...
66,T067,1,508.2,N056,N057,-2.128130,-79.949269,-2.123571,-79.949298,-2.125853,...,0.408497,0.615133,0.310370,0.753634,0.559712,Media,21,JUEVES,NO_IDENTIFICADO,Exposición media (0.56). 33 vehículos involucr...
26,T027,10,491.5,N007,N006,-2.079200,-79.931886,-2.079088,-79.927470,-2.079124,...,0.335784,0.238766,0.741521,0.600148,0.526812,Media,1,SÁBADO,AUTOMOVIL,Exposición media (0.53). 13 vehículos involucr...
23,T024,10,491.5,N010,N009,-2.084741,-79.943893,-2.082744,-79.939946,-2.083744,...,0.386878,0.198972,0.671885,0.707999,0.526285,Media,8,DOMINGO,NO_IDENTIFICADO,Exposición media (0.53). 11 vehículos involucr...
36,T037,9,391.3,N026,N027,-2.116081,-79.948310,-2.119559,-79.948960,-2.117815,...,0.295848,0.454956,0.589265,0.515847,0.520022,Media,19,SÁBADO,AUTOMOVIL,Exposición media (0.52). 19 vehículos involucr...
38,T039,1,508.2,N028,N029,-2.234834,-79.936169,-2.233628,-79.940578,-2.234226,...,0.251838,0.345726,0.757453,0.422949,0.508709,Media,19,SÁBADO,NO_IDENTIFICADO,Exposición media (0.51). 19 vehículos involucr...


In [26]:
mapa_dimension(datasetDim_E, 'valor_e', 'Exposición (E)', [
    ('E', 'valor_e', f2), ('Nivel', 'nivel_e', ft), ('Siniestros', 'n_siniestros', f0),
    ('Vehículos/km', 'e_vehiculos_km', lambda v: f'{v:.0f}'), ('Hora pico', 'casos_hora_pico', f0),
    ('Peatonales', 'casos_peatonal', f0), ('Vehículo más frecuente', 'vehiculo_mas_frecuente', ft),
    ('Nodos', 'nodo_inicio', ft), ('→', 'nodo_fin', ft), ('Descripción', 'descripcion_e', ft)],
    puntos=d[d['f_peatonal']], nombre_puntos='Siniestros peatonales')

## 9. Dimensión V — Vulnerabilidad → `datasetDim_V`
| Indicador | Condición en `Dataset_Limpio` |
|---|---|
| Letalidad | fallecidos por siniestro (suavizado) |
| Lesividad | lesionados por siniestro (suavizado) |
| Cruce sin protección | `codigo_causa` ∈ C16, C26, C27 |
| Usuarios vulnerables | `bicicleta`, `motocicleta`, `scooter_electrico` o `tricimoto` mayor que 0 |

In [27]:
d['f_cruce_inseguro'] = d['codigo_causa'].isin(['C16', 'C26', 'C27'])
d['f_usuario_vulnerable'] = d[['bicicleta', 'motocicleta', 'scooter_electrico', 'tricimoto']].sum(axis=1).gt(0)

V = base.copy()
V['casos_cruce_inseguro'] = conteo('f_cruce_inseguro'); V['casos_usuario_vulnerable'] = conteo('f_usuario_vulnerable')
V['v_letalidad'] = tasa(V['fallecidos'], d['fallecidos'].mean())
V['v_lesividad'] = tasa(V['lesionados'], d['lesionados'].mean())
V['v_cruce_inseguro'] = tasa(V['casos_cruce_inseguro'], d['f_cruce_inseguro'].mean())
V['v_usuario_vulnerable'] = tasa(V['casos_usuario_vulnerable'], d['f_usuario_vulnerable'].mean())
VC = ['v_letalidad', 'v_lesividad', 'v_cruce_inseguro', 'v_usuario_vulnerable']
for c in VC:
    V[c + '_n'] = minmax(V[c])
V['valor_v'] = V[[c + '_n' for c in VC]].mean(axis=1).where(CON, 0.0)
V['nivel_v'] = [nivel(v, n) for v, n in zip(V['valor_v'], V['n_siniestros'])]
V['atropellos'] = V['tramo'].map(d[d['tipo_siniestro'].eq('ATROPELLOS')].groupby('tramo').size()).fillna(0).astype(int)
V['motos'] = V['tramo'].map(d.groupby('tramo')['motocicleta'].sum()).fillna(0).astype(int)
V['bicicletas'] = V['tramo'].map(d.groupby('tramo')['bicicleta'].sum()).fillna(0).astype(int)
def desc_v(r):
    if r.n_siniestros == 0: return 'Sin siniestros registrados en 2024-2026.'
    return (f'Vulnerabilidad {r.nivel_v.lower()} ({r.valor_v:.2f}). {r.fallecidos} fallecidos y {r.lesionados} lesionados en '
            f'{r.n_siniestros} siniestros; {r.atropellos} atropellos, {r.casos_cruce_inseguro} por cruce sin protección y '
            f'{r.casos_usuario_vulnerable} con motos, bicicletas o scooters.')
V['descripcion_v'] = [desc_v(r) for r in V.itertuples()]
datasetDim_V = V
reemplazar('datasetDim_V', datasetDim_V, 'tramo')
datasetDim_V.drop(columns='geometria').sort_values('valor_v', ascending=False).head(10)

✔ datasetDim_V: 88 filas guardadas en Supabase


,tramo,pieza,longitud_m,nodo_inicio,nodo_fin,lat_inicio,lon_inicio,lat_fin,lon_fin,lat_centro,...,v_letalidad_n,v_lesividad_n,v_cruce_inseguro_n,v_usuario_vulnerable_n,valor_v,nivel_v,atropellos,motos,bicicletas,descripcion_v
31,T032,6,463.0,N022,N023,-2.118906,-79.948633,-2.123031,-79.949348,-2.120968,...,0.393085,0.761985,0.507702,0.511677,0.543612,Media,2,2,0,Vulnerabilidad media (0.54). 1 fallecidos y 8 ...
47,T048,1,508.2,N037,N038,-2.207488,-79.955451,-2.202993,-79.956355,-2.205215,...,0.279390,0.590848,0.062144,1.000000,0.483096,Media,0,6,0,Vulnerabilidad media (0.48). 1 fallecidos y 10...
74,T075,4,473.6,N066,N067,-2.237651,-79.927558,-2.236280,-79.931592,-2.236969,...,0.279390,0.516594,1.000000,0.133758,0.482435,Media,4,1,0,Vulnerabilidad media (0.48). 1 fallecidos y 9 ...
64,T065,1,508.2,N054,N055,-2.136901,-79.946536,-2.132528,-79.947945,-2.134716,...,0.376934,0.637296,0.206681,0.678774,0.474921,Media,3,16,0,Vulnerabilidad media (0.47). 6 fallecidos y 43...
79,T080,3,475.8,N065,N064,-2.239171,-79.923532,-2.240079,-79.919369,-2.239794,...,0.404039,0.368996,0.585748,0.516288,0.468767,Media,4,4,0,Vulnerabilidad media (0.47). 2 fallecidos y 9 ...
81,T082,2,478.4,N068,N069,-2.235190,-79.935763,-2.236563,-79.931685,-2.235873,...,0.662780,0.365963,0.090878,0.742675,0.465574,Media,1,3,0,Vulnerabilidad media (0.47). 2 fallecidos y 4 ...
34,T035,7,449.1,N023,N025,-2.123065,-79.949104,-2.119077,-79.948342,-2.121079,...,0.248794,0.328385,0.918397,0.340855,0.459108,Media,9,7,0,Vulnerabilidad media (0.46). 2 fallecidos y 15...
71,T072,4,473.6,N063,N064,-2.239516,-79.915151,-2.239928,-79.919380,-2.239840,...,1.000000,0.231491,0.344084,0.259775,0.458838,Media,2,3,0,Vulnerabilidad media (0.46). 5 fallecidos y 6 ...
77,T078,3,475.8,N067,N066,-2.236420,-79.931640,-2.237773,-79.927579,-2.237092,...,0.577414,0.241703,0.585748,0.367789,0.443163,Media,4,3,0,Vulnerabilidad media (0.44). 3 fallecidos y 7 ...
72,T073,4,473.6,N064,N065,-2.239928,-79.919380,-2.238999,-79.923517,-2.239638,...,0.464630,0.212715,0.698807,0.371000,0.436788,Media,6,4,0,Vulnerabilidad media (0.44). 3 fallecidos y 9 ...


In [ ]:
mapa_dimension(datasetDim_V, 'valor_v', 'Vulnerabilidad (V)', [
    ('V', 'valor_v', f2), ('Nivel', 'nivel_v', ft), ('Siniestros', 'n_siniestros', f0),
    ('Fallecidos', 'fallecidos', f0), ('Lesionados', 'lesionados', f0), ('Atropellos', 'atropellos', f0),
    ('Cruce sin protección', 'casos_cruce_inseguro', f0), ('Con motos/bicis', 'casos_usuario_vulnerable', f0),
    ('Nodos', 'nodo_inicio', ft), ('→', 'nodo_fin', ft), ('Descripción', 'descripcion_v', ft)],
    puntos=d[d['fallecidos'] > 0], nombre_puntos='Siniestros con fallecidos')

## 10. Mapa integrado: las tres dimensiones leídas desde Supabase
Se leen `datasetDim_H`, `datasetDim_E` y `datasetDim_V` directamente de Supabase, se unen por tramo y se calcula la criticidad, ponderada y multiplicada.
- **Control de capas:** cambia entre H, E, V, criticidad ponderada y criticidad multiplicada, y muestra u oculta nodos y siniestros.
- **Clic en un tramo:** abre la ficha completa con las tres dimensiones.

In [28]:
dh = leer('datasetDim_H', 'tramo'); de = leer('datasetDim_E', 'tramo'); dv = leer('datasetDim_V', 'tramo')
comunes = ['tramo', 'pieza', 'longitud_m', 'nodo_inicio', 'nodo_fin', 'lat_centro', 'lon_centro', 'geometria',
           'parroquia', 'referencia', 'n_siniestros', 'fallecidos', 'lesionados']
R = (dh[comunes + ['valor_h', 'nivel_h', 'descripcion_h']]
       .merge(de[['tramo', 'valor_e', 'nivel_e', 'descripcion_e']], on='tramo')
       .merge(dv[['tramo', 'valor_v', 'nivel_v', 'descripcion_v']], on='tramo'))
con = R['n_siniestros'] > 0
w = pd.Series(PESOS) / sum(PESOS.values())
R['crit_ponderada'] = (R.valor_h * w.H + R.valor_e * w.E + R.valor_v * w.V).where(con, 0.0)
prod = (0.1 + 0.9 * R.valor_h) * (0.1 + 0.9 * R.valor_e) * (0.1 + 0.9 * R.valor_v)
lo, hi = prod[con].min(), prod[con].max()
R['crit_multiplicada'] = ((prod - lo) / (hi - lo) if hi > lo else prod * 0).where(con, 0.0)
for c in ['crit_ponderada', 'crit_multiplicada']:
    R['nivel_' + c] = [nivel(v, n) for v, n in zip(R[c], R['n_siniestros'])]
print(f'{len(R)} tramos integrados desde Supabase')
display(R.drop(columns='geometria').sort_values('crit_ponderada', ascending=False).head(10).round(3))

88 tramos integrados desde Supabase


,tramo,pieza,longitud_m,nodo_inicio,nodo_fin,lat_centro,lon_centro,parroquia,referencia,n_siniestros,...,valor_e,nivel_e,descripcion_e,valor_v,nivel_v,descripcion_v,crit_ponderada,crit_multiplicada,nivel_crit_ponderada,nivel_crit_multiplicada
79,T080,3,475.8,N065,N064,-2.240,-79.921,XIMENA,AV. PERIMETRAL Y CALLE VICENTE ARNULFO ROSERO ...,9,...,0.493,Media,Exposición media (0.49). 11 vehículos involucr...,0.469,Media,Vulnerabilidad media (0.47). 2 fallecidos y 9 ...,0.478,1.000,Media,Alta
64,T065,1,508.2,N054,N055,-2.135,-79.947,TARQUI,"Vía Perimetral y Calle 18 NO, Pasando el Desta...",32,...,0.591,Media,Exposición media (0.59). 50 vehículos involucr...,0.475,Media,Vulnerabilidad media (0.47). 6 fallecidos y 43...,0.475,0.938,Media,Alta
87,T088,14,346.5,N074,N073,-2.239,-79.913,XIMENA,VIA PERIMETRAL (SOBRE EL PUENTE ELEVADO VEHICU...,10,...,0.446,Media,Exposición media (0.45). 17 vehículos involucr...,0.435,Media,Vulnerabilidad media (0.43). 3 fallecidos y 9 ...,0.471,0.949,Media,Alta
81,T082,2,478.4,N068,N069,-2.236,-79.934,XIMENA,"VIA PERIMETRAL Y AV. 34 S-O, ALTURA DEL TUTI.",4,...,0.354,Media,Exposición media (0.35). 5 vehículos involucra...,0.466,Media,Vulnerabilidad media (0.47). 2 fallecidos y 4 ...,0.466,0.886,Media,Alta
72,T073,4,473.6,N064,N065,-2.240,-79.921,XIMENA,Av. Perimetral y 3° Pasaje 25 - Sector Isla Tr...,12,...,0.503,Media,Exposición media (0.50). 14 vehículos involucr...,0.437,Media,Vulnerabilidad media (0.44). 3 fallecidos y 9 ...,0.460,0.891,Media,Alta
31,T032,6,463.0,N022,N023,-2.121,-79.949,PASCUALES,Vía Perimetral y calle sin nombre (sobre el pa...,4,...,0.369,Media,Exposición media (0.37). 6 vehículos involucra...,0.544,Media,Vulnerabilidad media (0.54). 1 fallecidos y 8 ...,0.459,0.862,Media,Alta
85,T086,2,478.4,N072,N063,-2.240,-79.917,XIMENA,Vía Perimetral y calle Germán Antonio Pituir. ...,5,...,0.588,Media,Exposición media (0.59). 5 vehículos involucra...,0.290,Baja,Vulnerabilidad baja (0.29). 0 fallecidos y 7 l...,0.454,0.785,Media,Alta
73,T074,4,473.6,N065,N066,-2.238,-79.926,XIMENA,Av. Perimetral y Av.29 - Sector Isla Trinitaria.,22,...,0.498,Media,Exposición media (0.50). 35 vehículos involucr...,0.360,Media,Vulnerabilidad media (0.36). 2 fallecidos y 29...,0.451,0.823,Media,Alta
20,T021,10,491.5,N013,N012,-2.094,-79.948,PASCUALES,VÍA PERIMETRAL Y CALLE MARCEL LANIADO DE WIND,10,...,0.479,Media,Exposición media (0.48). 15 vehículos involucr...,0.412,Media,Vulnerabilidad media (0.41). 3 fallecidos y 14...,0.440,0.781,Media,Alta
34,T035,7,449.1,N023,N025,-2.121,-79.949,PASCUALES,VIA PERIMETRAL Y 2DO. CALLEJON 21 NO.,16,...,0.620,Media,Exposición media (0.62). 20 vehículos involucr...,0.459,Media,Vulnerabilidad media (0.46). 2 fallecidos y 15...,0.434,0.623,Media,Media


In [29]:
# Mapa base CARTO con API key (respaldo por si no se ejecutó la celda de utilidades)
if 'TILES_CARTO' not in globals():
    CARTO_KEY = 'cb1_5do9_1_90e38ab2e2043b7ab55c5b13'
    TILES_CARTO = ('https://basemaps.cartocdn.com/rastertiles/light_all/'
                   '{z}/{x}/{y}.png?key=' + CARTO_KEY)
    ATTR_CARTO = '© OpenStreetMap contributors © CARTO'

cmap = branca.colormap.LinearColormap(['#1a9850', '#fee08b', '#d73027'], vmin=0, vmax=1, caption='Valor (0 = bajo, 1 = alto)')
def barra(v):
    return (f'<div style="background:#eee;width:110px;height:9px;display:inline-block"><div style="background:{cmap(v)};'
            f'width:{v*110:.0f}px;height:9px"></div></div> {v:.2f}')
def ficha(r):
    return f'''<div style="font-family:sans-serif;font-size:12px;width:360px">
    <b style="font-size:14px">{r.tramo}</b> · {r.parroquia} · {r.longitud_m:.0f} m · nodos {r.nodo_inicio} → {r.nodo_fin}<br>
    <i>{txt(r.referencia, 120)}</i><hr style="margin:4px 0">
    <b>Siniestros:</b> {r.n_siniestros} · <b>Fallecidos:</b> {r.fallecidos} · <b>Lesionados:</b> {r.lesionados}<br>
    <b>H</b> {barra(r.valor_h)} {r.nivel_h}<br><b>E</b> {barra(r.valor_e)} {r.nivel_e}<br><b>V</b> {barra(r.valor_v)} {r.nivel_v}<hr style="margin:4px 0">
    <b>Criticidad ponderada:</b> {r.crit_ponderada:.2f} ({r.nivel_crit_ponderada})<br>
    <b>Criticidad multiplicada:</b> {r.crit_multiplicada:.2f} ({r.nivel_crit_multiplicada})<hr style="margin:4px 0">
    <b>H:</b> {r.descripcion_h}<br><b>E:</b> {r.descripcion_e}<br><b>V:</b> {r.descripcion_v}</div>'''

mapa = folium.Map(location=[R.lat_centro.mean(), R.lon_centro.mean()],
                  zoom_start=13, tiles=TILES_CARTO, attr=ATTR_CARTO)
for col, nombre, ver in [('crit_ponderada', 'Criticidad · promedio ponderado', True),
                         ('crit_multiplicada', 'Criticidad · multiplicación', False),
                         ('valor_h', 'H · Peligrosidad', False), ('valor_e', 'E · Exposición', False),
                         ('valor_v', 'V · Vulnerabilidad', False)]:
    fg = folium.FeatureGroup(name=nombre, show=ver)
    for r in R.itertuples():
        color = '#9e9e9e' if r.n_siniestros == 0 else cmap(getattr(r, col))
        folium.GeoJson(r.geometria, style_function=lambda f, c=color: {'color': c, 'weight': 7, 'opacity': .9},
                       highlight_function=lambda f: {'weight': 12, 'color': '#000'},
                       tooltip=f'{r.tramo} · {nombre}: {getattr(r, col):.2f}',
                       popup=folium.Popup(ficha(r), max_width=400)).add_to(fg)
    fg.add_to(mapa)
fn = folium.FeatureGroup(name='Nodos', show=True)
for n in nodos.to_crs(4326).itertuples():
    folium.CircleMarker([n.geometry.y, n.geometry.x], radius=3 + n.grado, color='#253494', fill=True, fill_opacity=.9,
                        tooltip=f'Nodo {n.nodo} · grado {n.grado}').add_to(fn)
fn.add_to(mapa)
fs = folium.FeatureGroup(name='Siniestros (Dataset_Limpio)', show=False)
for s in gdl.to_crs(4326).itertuples():
    col = '#000' if s.fallecidos > 0 else ('#e66101' if s.lesionados > 0 else '#5e3c99')
    folium.CircleMarker([s.geometry.y, s.geometry.x], radius=3, color=col, fill=True, fill_opacity=.8,
                        tooltip=f'{s.id_siniestro} · {s.tipo_siniestro} · {s.fecha_hora:%Y-%m-%d %H:%M}').add_to(fs)
fs.add_to(mapa)
cmap.add_to(mapa)
folium.LayerControl(collapsed=False).add_to(mapa)
mapa.save('mapa_integrado_HEV.html')
mapa

In [ ]:
print('Resumen por nivel (criticidad ponderada):', R['nivel_crit_ponderada'].value_counts().to_dict())
print('Resumen por nivel (criticidad multiplicada):', R['nivel_crit_multiplicada'].value_counts().to_dict())
print('Mapa guardado en el panel de archivos: mapa_integrado_HEV.html')

Resumen por nivel (criticidad ponderada): {'Media': 62, 'Baja': 24, 'Sin registros': 2}
Resumen por nivel (criticidad multiplicada): {'Media': 38, 'Baja': 35, 'Alta': 13, 'Sin registros': 2}
Mapa guardado en el panel de archivos: mapa_integrado_HEV.html
